In [1]:
# Import necessary libraries
import json
import os
from typing import Any, Dict

import boto3
import requests
from dotenv import load_dotenv

load_dotenv()
print("We are up and running!")

We are up and running!


AWS credentials are picked up automatically from the environment (instance profile, env vars, or `~/.aws/credentials`).

In [2]:
MODEL_ID = "eu.amazon.nova-lite-v1:0"
REGION = "eu-west-1"

bedrock = boto3.client("bedrock-runtime", region_name=REGION)

print(f"Bedrock client ready — model: {MODEL_ID}")

Bedrock client ready — model: eu.amazon.nova-lite-v1:0


In [3]:
PROMPT = "What is the current price of Bitcoin?"

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=[
        {"role": "user", "content": [{"text": PROMPT}]}
    ],
)

print(response["output"]["message"]["content"][0]["text"])

I'm unable to provide real-time data as my current information is only up to 2022 and I don't have access to external sources for live updates. To find the current price of Bitcoin, you can check financial news websites, cryptocurrency exchanges, or dedicated platforms like CoinMarketCap or CoinGecko. These sources typically provide up-to-date information on the price of Bitcoin and other cryptocurrencies.


In [4]:
# Inspect the full raw response from Bedrock
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "1f1ef02f-6350-4896-ade7-c2742fca19c9",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:06:48 GMT",
      "content-type": "application/json",
      "content-length": "610",
      "connection": "keep-alive",
      "x-amzn-requestid": "1f1ef02f-6350-4896-ade7-c2742fca19c9"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "I'm unable to provide real-time data as my current information is only up to 2022 and I don't have access to external sources for live updates. To find the current price of Bitcoin, you can check financial news websites, cryptocurrency exchanges, or dedicated platforms like CoinMarketCap or CoinGecko. These sources typically provide up-to-date information on the price of Bitcoin and other cryptocurrencies."
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 8,
    "o

In [5]:
url = "https://api.binance.com/api/v3/ticker/price?symbol=BTCUSDT"
response = requests.get(url)
data = response.json()
print(data)

{'symbol': 'BTCUSDT', 'price': '66120.31000000'}


In [6]:
# Define the function
def get_crypto_price(symbol: str) -> float:
    """
    Get the current price of a cryptocurrency from Binance API
    """
    url = f"https://api.binance.com/api/v3/ticker/price?symbol={symbol}"
    response = requests.get(url)
    data = response.json()
    return float(data["price"])

In [7]:
price = get_crypto_price("BTCUSDT")
print(f"BTC Price in USDT: {price}")

BTC Price in USDT: 66115.02


In [8]:
# Tool definition in Bedrock Converse format
tool_config = {
    "tools": [
        {
            "toolSpec": {
                "name": "get_crypto_price",
                "description": "Get cryptocurrency price in USDT from Binance",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol "
                                    "(e.g., BTCUSDT, ETHUSDT). "
                                    "The symbol for Bitcoin is BTCUSDT. "
                                    "The symbol for Ethereum is ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [14]:
# in-class exercise
# Tool definition in Bedrock Converse format
tool_config2 = {
    "tools": [
        {
            "toolSpec": {
                "name": "execute_python_function",
                "description": "Execute any python code for the user",
                "inputSchema": {
                    "json": {
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The python code"
                                ),
                            }
                        },
                        "required": ["symbol"],
                    }
                },
            }
        }
    ]
}

In [17]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config2,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "e66a81f6-84b5-4b97-9dab-7bcda6bfba2e",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:30:13 GMT",
      "content-type": "application/json",
      "content-length": "744",
      "connection": "keep-alive",
      "x-amzn-requestid": "e66a81f6-84b5-4b97-9dab-7bcda6bfba2e"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking>I can't directly access real-time data such as the current price of Bitcoin. However, I can execute a Python function that fetches this information from an online source. I will need to use a tool that can access the internet and retrieve data from a financial API.</thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_gcbECqqE4ZNYxwMnQ7hOQB",
            "name": "execute_python_function",
            "input": {
              "symbol": "import requests; resp

In [9]:
PROMPT = "What is the current price of Bitcoin?"

messages = [
    {"role": "user", "content": [{"text": PROMPT}]}
]

response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

# Print the full raw response so we can see the tool call request
print(json.dumps(response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "7ffeb4cb-9c5b-40d6-b9ab-cce2356248da",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:06:50 GMT",
      "content-type": "application/json",
      "content-length": "495",
      "connection": "keep-alive",
      "x-amzn-requestid": "7ffeb4cb-9c5b-40d6-b9ab-cce2356248da"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking> The User has asked for the current price of Bitcoin. I can use the \"get_crypto_price\" tool to get this information. The symbol for Bitcoin is BTCUSDT. </thinking>\n"
        },
        {
          "toolUse": {
            "toolUseId": "tooluse_l0bnGiyvLcU6MGM1xT78Ux",
            "name": "get_crypto_price",
            "input": {
              "symbol": "BTCUSDT"
            }
          }
        }
      ]
    }
  },
  "stopReason": "tool_use",
  "usage": {
    "inputTokens": 444,
    "outpu

In [10]:
# Extract the tool call from the response
assistant_message = response["output"]["message"]
tool_use_block = next(
    block for block in assistant_message["content"] if "toolUse" in block
)

tool_use_id = tool_use_block["toolUse"]["toolUseId"]
tool_name = tool_use_block["toolUse"]["name"]
tool_input = tool_use_block["toolUse"]["input"]

print(f"Tool requested: {tool_name}")
print(f"Tool input:     {tool_input}")
print(f"Tool use ID:    {tool_use_id}")

Tool requested: get_crypto_price
Tool input:     {'symbol': 'BTCUSDT'}
Tool use ID:    tooluse_l0bnGiyvLcU6MGM1xT78Ux


In [11]:
# Manually call the function with the arguments the model provided
price = get_crypto_price(**tool_input)
print(f"Result from function: {price}")

Result from function: 66117.73


In [12]:
# Build the full conversation: original user message + assistant tool call + our tool result
messages = [
    {"role": "user", "content": [{"text": PROMPT}]},
    assistant_message,  # the model's response containing the toolUse block
    {
        "role": "user",
        "content": [
            {
                "toolResult": {
                    "toolUseId": tool_use_id,
                    "content": [{"text": str(price)}],
                }
            }
        ],
    },
]

final_response = bedrock.converse(
    modelId=MODEL_ID,
    messages=messages,
    toolConfig=tool_config,
)

print(json.dumps(final_response, indent=2, default=str))

{
  "ResponseMetadata": {
    "RequestId": "44082077-ee89-4f96-9529-ea35b01cd750",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Mon, 02 Mar 2026 11:06:52 GMT",
      "content-type": "application/json",
      "content-length": "420",
      "connection": "keep-alive",
      "x-amzn-requestid": "44082077-ee89-4f96-9529-ea35b01cd750"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "<thinking> I have used the \"get_crypto_price\" tool and got the result. The current price of Bitcoin (BTC) is 66117.73 USDT. </thinking>\n\n<response> The current price of Bitcoin (BTC) is 66117.73 USDT. </response>"
        }
      ]
    }
  },
  "stopReason": "end_turn",
  "usage": {
    "inputTokens": 544,
    "outputTokens": 71,
    "totalTokens": 615
  },
  "metrics": {
    "latencyMs": 659
  }
}


In [13]:
import re

text = final_response["output"]["message"]["content"][0]["text"]
clean_text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL | re.IGNORECASE).strip()

print(clean_text)

<response> The current price of Bitcoin (BTC) is 66117.73 USDT. </response>
